# Reproducible stock-movement experiment: text availability versus lexical content

**Study:** *Separating Text Availability and Lexical Information in Stock-Movement Prediction for Indonesian Equities*  
**Execution environments:** Google Colab, Kaggle Notebooks, or local Jupyter/Python.

### Research aim and estimand

The primary question is whether archived Indonesian-language tweet **content** improves next-session three-class stock-movement classification **after the model already observes whether tweets are present**. The key estimand is the paired difference in 2024 Macro-F1 between a lexical model and an availability-matched control on exactly the same ticker–session observations. Report log loss, multiclass Brier score, class-level results, and calibration alongside Macro-F1, since classification and probability quality need not agree.

### Data and partition rules

- **Only two observed source files:** `Stock Market Data(2).csv` and `Tweet Data - ID(2).csv`, with alternative filenames accepted only if their byte-level SHA-256 values match the pinned originals. No external data or synthetic observations are introduced.
- **Source scope:** 15 stocks, 10,830 observed market ticker–sessions, and 7,672 archived tweets from 2022–2024. The source includes 201 tweets without an assigned trading session; these are quarantined rather than assigned an invented date.
- **Model development:** eligible 2022 observations train initial candidates; 2023 observations select hyperparameters. The final models are refitted on eligible 2022–2023 observations.
- **Chronological evaluation:** 2024 observations are evaluated after development; the 2024 labels do not inform parameter selection, text-vocabulary construction, scaling, imputation, or final model fitting *in this execution*. Chronological separation should not be described as external validation or evidence of a never-inspected historical period.
- **Transfer stress test:** three fixed, five-stock holdout groups test how performance changes when each evaluation stock is excluded from supervised fitting and model selection.

### Output and interpretation policy

This notebook exports **20 numbered analytical CSV tables and 20 numbered PNG figures (300 dpi)**, then creates a single ZIP with only those 40 outputs. Figure counts and summary statistics must be grounded in the loaded sources and actual model predictions. Observed improvements are descriptive until uncertainty tests support a stronger conclusion. A reproducible workflow does not itself prove model superiority, causality, investment returns, external validity, or acceptance in a highly ranked journal.

**Reproducibility note:** Figure and table inputs come from transformations computed during the current run. SHA-256 and source-integrity checks protect against silent substitution of dataset versions. Refer to the repository's `docs/` files for licence, provenance, version sensitivity, and archived-result reconciliation.

In [ ]:
# Load dependencies and establish a deterministic seed for stochastic components.
from pathlib import Path
import os, sys, hashlib, zipfile, math, platform, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score, log_loss,
                             confusion_matrix, precision_recall_fscore_support)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display
from sklearn import __version__ as SKLEARN_VERSION

# Pin class order, text-vectorisation budgets, and the input checksums before modelling.
SEED = 20261024
np.random.seed(SEED)
LABELS = ['Down','Flat','Up']
MODEL_NAMES = ['M_market','A_availability','W_word','C_character','H_hybrid']
ALL_MODELS=MODEL_NAMES+['F_extra_trees']
MODEL_LABELS = {'M_market':'Market', 'A_availability':'Market + availability',
                'W_word':'Word TF-IDF', 'C_character':'Character TF-IDF',
                'H_hybrid':'Hybrid TF-IDF','F_extra_trees':'Extra Trees + availability'}
COLORS = ['#0072B2','#E69F00','#009E73','#CC79A7','#56B4E9','#444444']
EPS=1e-10
BS_REPS=2000
CS=[0.01,0.1,1.0]
EXPECTED_MARKET_SHA='51c847f70cd310806b90a62dd551779e9ef43480a07714a5189fc884f43aaff6'
EXPECTED_TWEET_SHA='4c513609af0342ed968f3ee26381a823bf158ca2661ad2b8894c2989aa669deb'
# Use platform-appropriate paths; generated artifacts remain separate from committed results.
IS_KAGGLE=Path('/kaggle/working').exists()
IS_COLAB=(bool(os.environ.get('COLAB_RELEASE_TAG')) or 'google.colab' in sys.modules)
REPO_ROOT = Path.cwd() if (Path.cwd()/'README.md').exists() else (Path.cwd().parent if (Path.cwd().parent/'README.md').exists() else Path.cwd())
OUT_BASE=Path('/kaggle/working' if IS_KAGGLE else '/content' if IS_COLAB else os.environ.get('STOCK_OUTPUT_DIR',str(REPO_ROOT/'artifacts')))
RESULTS=OUT_BASE/'Q1_20PNG_20CSV'
FIG=RESULTS/'figures'
TAB=RESULTS/'tables'
# Remove stale outputs so each ZIP reflects only the current execution.
for root in [FIG,TAB]:
    root.mkdir(parents=True,exist_ok=True)
    for old in root.iterdir():
        if old.is_file() and old.suffix.lower() in {'.png','.csv'}:
            old.unlink()
ZIP_FILE=OUT_BASE/'Q1_Results_20PNG_20CSV.zip'
if ZIP_FILE.exists(): ZIP_FILE.unlink()
# Configure consistent, publication-oriented plot defaults at 300 dpi.
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,
                     'axes.spines.right':False,'savefig.dpi':300,'axes.titleweight':'bold'})

## 1. Identify and authenticate the original sources

**Objective.** Make the experimental population explicit before any feature processing. The input search accepts either a local repository `data/raw/` directory, uploaded Colab files, Kaggle input files, or explicit environment-variable paths. SHA-256 checks establish *byte-for-byte* correspondence with the declared source archive; a source mismatch raises an exception rather than merging or substituting data silently.

**Required checks.** Confirm the OHLCV schema, unique ticker–date market keys, positive and internally consistent prices, nonnegative trading volume, and the declared record counts. Split archived posts using `trading_session_date`, **not** their publication calendar year. The 15 posts published in December 2023 but assigned to a January 2024 trading session belong to 2024 text observations. Keep the 201 posts without an assigned session outside the model inputs.

**Interpretive boundary.** The variable `has_archived_text` means that the *supplied dataset* contains at least one session-assigned tweet. It does not demonstrate that all relevant social-media activity was captured, or that no discussion occurred when the archive has no assigned post. Source-file hash matching confirms identity, not completeness or ground truth.

In [ ]:
# SHA-256 identifies the exact source bytes and prevents unlogged file substitutions.
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

# Search approved local, Colab and Kaggle input directories; never fetch third-party data silently.
def find_sources(prompt_upload=True):
    candidates=[]
    env_paths=[os.environ.get('STOCK_MARKET_CSV',''),os.environ.get('STOCK_TWEET_CSV','')]
    candidates.extend(Path(p) for p in env_paths if p and Path(p).is_file())
    for base in [REPO_ROOT/'data/raw', Path('/content'), Path('/kaggle/input')]:
        if base.exists():
            for x in base.rglob('*.csv'):
                if len(x.relative_to(base).parts)<=5: candidates.append(x)
    def pick(role, sha, starts):
        hit=[p for p in candidates if p.name.lower().startswith(starts) and sha256(p)==sha]
        if not hit:return None
        return sorted(hit,key=lambda x:(len(x.parts),len(str(x))))[0]
    market=pick('market',EXPECTED_MARKET_SHA,'stock market data')
    tweet=pick('tweets',EXPECTED_TWEET_SHA,'tweet data - id')
    if (market is None or tweet is None) and IS_COLAB and prompt_upload:
        from google.colab import files
        print('Upload the TWO original CSV source files.')
        files.upload()
        return find_sources(prompt_upload=False)
    if market is None or tweet is None:
        raise FileNotFoundError('Both original CSV sources with the verified hashes are required. '
              'Attach them to Kaggle or upload to Colab. Current files were not silently replaced.')
    return market,tweet

# Validate both input schemas, OHLCV constraints and observed population sizes before merging.
MARKET_FILE, TWEET_FILE=find_sources()
market=pd.read_csv(MARKET_FILE,parse_dates=['date'])
tweets=pd.read_csv(TWEET_FILE,parse_dates=['created_at','trading_session_date'])
assert list(market.columns)==['ticker','company_name','date','open','high','low','close','volume']
assert list(tweets.columns)==['ticker','full_text','created_at','trading_session_date','favorite_count','retweet_count','reply_count','is_quote_status','sentiment']
assert len(market)==10830 and len(tweets)==7672 and market.ticker.nunique()==15
assert market[['ticker','date']].duplicated().sum()==0
assert not market[['open','high','low','close','volume']].isna().any().any()
assert market[['open','high','low','close']].gt(0).all().all()
assert not (market['volume']<0).any()
assert (market['high']>=market[['low','open','close']].max(axis=1)-1e-5).all()
assert (market['low']<=market[['high','open','close']].min(axis=1)+1e-5).all()
market=market.sort_values(['ticker','date']).reset_index(drop=True)
# Assign posts by trading-session date; quarantine those without a valid market session.
assigned=tweets[tweets.trading_session_date.notna()].copy()
unassigned=tweets[tweets.trading_session_date.isna()].copy()
assert len(assigned)==7471 and len(unassigned)==201
assert not assigned[['ticker','trading_session_date']].drop_duplicates().merge(
    market[['ticker','date']].rename(columns={'date':'trading_session_date'}),on=['ticker','trading_session_date'],
    how='left',indicator=True).eval('_merge != "both"').any()
assert (assigned.created_at<=assigned.trading_session_date+pd.Timedelta(days=7)).all(), 'Check unusual future-post timestamps'
assert assigned[assigned.trading_session_date.dt.year.isin([2022,2023])].shape[0]==2562
assert assigned[assigned.trading_session_date.dt.year.eq(2024)].shape[0]==4909
# Preserve year-boundary posts as 2024 inputs when their assigned session is in 2024.
cross_year=assigned[(assigned.created_at.dt.year==2023)&(assigned.trading_session_date.dt.year==2024)]
assert len(cross_year)==15
# Merge counts and concatenated lexical documents on unique ticker–session keys.
counts=assigned.groupby(['ticker','trading_session_date']).size().rename('tweet_count').reset_index().rename(columns={'trading_session_date':'date'})
docs=assigned.groupby(['ticker','trading_session_date'],sort=True)['full_text'].agg(' '.join).rename('text_document').reset_index().rename(columns={'trading_session_date':'date'})
market=market.merge(counts,on=['ticker','date'],how='left',validate='one_to_one')
market=market.merge(docs,on=['ticker','date'],how='left',validate='one_to_one')
market['tweet_count']=market.tweet_count.fillna(0).astype(int)
market['has_archived_text']=(market.tweet_count>0).astype(int)
market['text_document']=market.text_document.fillna('')
assert int(market.tweet_count.sum())==7471
assert market.loc[market.tweet_count.eq(0),'text_document'].eq('').all()
print('SOURCE VERIFIED',{'market':len(market),'tweets':len(tweets),'assigned':len(assigned),'quarantine':len(unassigned),
                        'tickers':market.ticker.nunique(),'crossyear_assigned_to_2024':len(cross_year)})

## 2. Construct time-ordered predictors and next-session targets

**Prediction horizon.** For stock *i* at session *t*, define the realised next-session return as `close[i,t+1] / close[i,t] - 1`. Map returns less than `-1e-10` to **Down**, greater than `1e-10` to **Up**, and the numerical-zero interval to **Flat**. This near-zero rule is a floating-point tolerance, **not** an economically selected no-trade interval. Table 04 later examines how wider hypothetical thresholds change the target distribution.

**Information boundary.** Market predictors use the current or previous observed OHLCV records. Current text indicators use posts assigned to the current session; lagged text statistics use earlier sessions only. The design presumes that all features assigned to session *t* are available by the chosen prediction cutoff. This notebook checks trading-session assignment, but it does not independently audit original exchange cutoffs, tweet ingestion delays, vendor timestamps, or intraday release latency. That operational timing remains a limitation.

**Eligible observations.** Exclude the first 20 trading sessions for feature warm-up and the final session of each calendar year if the next observed session lies in a new year. Expected eligible counts are 3,375 for 2022, 3,570 for 2023, and 3,540 for 2024. The target is built for supervised evaluation only; future closing values must never enter the predictor matrix.

In [ ]:
# Include calendar covariates in the market block; availability covariates form a separate block.
MARKET_FEATS=['return_1','return_5','return_10','return_20','intraday_range','close_vs_open',
              'log_volume','relative_volume20','volatility10','rsi14','zero_volume',
              'month_sin','month_cos','weekday']
AVAIL_FEATS=['current_text','current_tweet_log','prior_text_share20','prior_tweet_mean20','since_last_text20']
FEATURES=MARKET_FEATS+AVAIL_FEATS

# All numeric market and availability predictors use current and preceding observed sessions.
def engineer(g):
    g=g.sort_values('date').copy()
    cl=g.close.astype(float);op=g.open.astype(float)
    for k in (1,5,10,20):g[f'return_{k}']=cl.pct_change(k,fill_method=None)
    g['intraday_range']=(g.high-g.low)/cl
    g['close_vs_open']=(cl-op)/op
    g['log_volume']=np.log1p(g.volume)
    histvol=g.volume.shift(1).rolling(20,min_periods=10).mean()
    g['relative_volume20']=g.volume/histvol.replace(0,np.nan)
    g['volatility10']=g.return_1.rolling(10,min_periods=5).std()
    diff=cl.diff();up=diff.clip(lower=0).rolling(14,min_periods=14).mean()
    down=(-diff.clip(upper=0)).rolling(14,min_periods=14).mean()
    rs=up/down.replace(0,np.nan)
    g['rsi14']=(100-100/(1+rs)).fillna(pd.Series(np.where((down==0)&(up>0),100,50),index=g.index))
    g['zero_volume']=(g.volume==0).astype(int)
    g['month_sin']=np.sin(2*np.pi*g.date.dt.month/12)
    g['month_cos']=np.cos(2*np.pi*g.date.dt.month/12)
    g['weekday']=g.date.dt.weekday.astype(int)
    g['current_text']=g.has_archived_text
    g['current_tweet_log']=np.log1p(g.tweet_count)
    g['prior_text_share20']=g.has_archived_text.shift(1).rolling(20,min_periods=1).mean()
    g['prior_tweet_mean20']=g.tweet_count.shift(1).rolling(20,min_periods=1).mean()
    gaps=[];gap=20
    for n in g.has_archived_text:
        gaps.append(min(gap,20))
        gap=1 if n else min(20,gap+1)
    g['since_last_text20']=gaps
# Use the next closing price only to construct the target; do not include it in FEATURES.
    g['next_date']=g.date.shift(-1)
    g['next_close']=cl.shift(-1)
    g['next_return']=g.next_close/cl-1
    g['target_label']=np.select([g.next_return < -EPS,g.next_return>EPS],['Down','Up'],default='Flat')
    g['position']=np.arange(len(g))
    g['year']=g.date.dt.year.astype(int)
# Drop warm-up sessions and within-year targets that cross a calendar boundary.
    g['eligible']=(g.position>=20)&g.next_date.notna()&(g.year==g.next_date.dt.year)
    return g
# Development and evaluation partitions are strictly chronological for this run.
panel=pd.concat([engineer(g) for _,g in market.groupby('ticker',sort=True)],ignore_index=True)
data=panel.loc[panel.eligible].sort_values(['date','ticker']).reset_index(drop=True)
train22=data.loc[data.year==2022].copy()
val23=data.loc[data.year==2023].copy()
dev=data.loc[data.year.isin([2022,2023])].copy()
test=data.loc[data.year==2024].copy()
assert (len(train22),len(val23),len(dev),len(test))==(3375,3570,6945,3540)
assert test[['ticker','date']].duplicated().sum()==0
assert dev.date.max()<test.date.min()
assert all(test.groupby('ticker').size()==236)
assert [data.loc[data.year.eq(y),'target_label'].value_counts().reindex(LABELS).tolist() for y in [2022,2023,2024]]==[[1545,591,1239],[1542,699,1329],[1397,705,1438]]
assert not any('next_' in col or 'target' in col for col in FEATURES)
print('ELIGIBILITY VERIFIED',{'2022':len(train22),'2023':len(val23),'development':len(dev),'test_2024':len(test)})

## 3. Fit transformations on development data and select models without 2024 feedback

**Matched comparison.** Five multinomial logistic-regression specifications share the same estimator family. `M_market` uses market and calendar variables plus ticker indicators; `A_availability` adds current and historical archive-presence variables. `W_word`, `C_character`, and `H_hybrid` each retain the complete availability-control input and add a fixed **16-component lexical block**: 16 word components, 16 character components, or 8 + 8 hybrid components respectively. Equal nominal dimensions do not imply equal information content.

**Train-only processing.** Numeric winsorisation bounds, median imputation, standardisation, word/character TF-IDF vocabularies, and Truncated SVD transformations are fitted on the permitted training subset. The corresponding 2023 or 2024 inputs are *transformed only*. Texts from the test period cannot alter vocabulary or SVD directions. Empty archived-text sessions are represented consistently without fabricating tweet content.

**Development selection.** Candidate logistic regularisation values are `C = {0.01, 0.1, 1.0}`. The notebook trains the candidates on 2022, scores 2023 Macro-F1, and selects a **single shared C** using the mean development score across the five matched logistic models. The selected value is then used for the full 2022–2023 refit. The nonlinear Extra Trees comparator uses the same market-plus-availability features without lexical content but selects its leaf-size parameter separately on 2023. Because its estimator differs, contrasts against Extra Trees are descriptive, not controlled estimates of added lexical information.

**Primary metrics.** Macro-F1 averages per-class F1 equally; balanced accuracy averages per-class recall. Lower multiclass log loss and Brier score indicate more informative probabilities. The Brier implementation uses the **sum across all three classes**, averaged over observations; state this convention when comparing to literature using a different normalisation.

In [ ]:
# Fit each preprocessing object on the permitted training partition only.
class InputFactory:
    def __init__(self,with_ticker=True):self.with_ticker=with_ticker
    def fit(self,tr):
# Compute winsorisation limits, imputation and standardisation from training rows.
        self.tickers=sorted(tr.ticker.unique())
        self.qlo=tr[FEATURES].replace([np.inf,-np.inf],np.nan).quantile(.005)
        self.qhi=tr[FEATURES].replace([np.inf,-np.inf],np.nan).quantile(.995)
        raw=tr[FEATURES].replace([np.inf,-np.inf],np.nan).clip(lower=self.qlo,upper=self.qhi,axis=1)
        self.imp=SimpleImputer(strategy='median',keep_empty_features=True)
        self.scale=StandardScaler()
        self.scale.fit(self.imp.fit_transform(raw))
# Fit vocabulary and SVD separately for word and character representations.
        self.words={}
        for key in ['word','character']:
            tf=(TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=7000,sublinear_tf=True,
                                token_pattern=r'(?u)\b\w\w+\b',dtype=np.float32)
                if key=='word' else TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),
                                min_df=2,max_features=7000,sublinear_tf=True,dtype=np.float32))
            z=tf.fit_transform(tr.text_document.fillna('').tolist())
            dim=min(16,z.shape[0]-1,z.shape[1]-1)
            if dim<8:raise ValueError('Insufficient lexical vocabulary for an 8-dimensional hybrid')
            svd=TruncatedSVD(n_components=dim,n_iter=4,random_state=SEED)
            vz=svd.fit_transform(z)
            sd=np.maximum(vz.std(axis=0),1e-6)
            self.words[key]=(tf,svd,sd)
        return self
# Transform validation or evaluation rows without learning from those rows.
    def transform(self,frame,mask_text=False):
        raw=frame[FEATURES].replace([np.inf,-np.inf],np.nan).clip(lower=self.qlo,upper=self.qhi,axis=1)
        v=self.scale.transform(self.imp.transform(raw))
        a=csr_matrix(v[:,:len(MARKET_FEATS)])
        b=csr_matrix(v)
        if self.with_ticker:
            ticks=pd.get_dummies(pd.Categorical(frame.ticker,categories=self.tickers),dtype=np.float64).to_numpy()
            tt=csr_matrix(ticks)
            a=hstack([a,tt],format='csr');b=hstack([b,tt],format='csr')
        docs=['']*len(frame) if mask_text else frame.text_document.fillna('').tolist()
        word,char={},{}
        blocks={}
        for key in ['word','character']:
            tf,svd,sd=self.words[key]
            emb=svd.transform(tf.transform(docs))/sd
            blocks[key]=csr_matrix(emb)
        return {'M_market':a,'A_availability':b,
                'W_word':hstack([b,blocks['word']],format='csr'),
                'C_character':hstack([b,blocks['character']],format='csr'),
                'H_hybrid':hstack([b,blocks['word'][:,:8],blocks['character'][:,:8]],format='csr')}

# Multiclass logistic regression uses the same estimator specification for all matched ablations.
def fit_model(x,y,c):
    clf=LogisticRegression(C=float(c),max_iter=1400,solver='lbfgs',random_state=SEED)
    clf.fit(x,np.asarray(y))
    assert list(clf.classes_)==LABELS
    return clf

def predict(clf,x):
    p=clf.predict_proba(x)
    yp=np.array(LABELS)[p.argmax(axis=1)]
    assert np.isfinite(p).all() and np.max(np.abs(p.sum(axis=1)-1))<1e-9
    return yp,p

# Multiclass Brier = mean over sessions of the sum of squared class-probability errors.
def score(y,yp,p):
    y=np.asarray(y);yp=np.asarray(yp)
    ix=pd.Categorical(y,categories=LABELS).codes
    assert ix.min()>=0 and len(y)==len(p)
    br=float(np.sum((p-np.eye(3)[ix])**2,axis=1).mean())
    return {'n':len(y),'macro_f1':float(f1_score(y,yp,average='macro',labels=LABELS,zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(y,yp)),
            'accuracy':float(accuracy_score(y,yp)),
            'log_loss':float(log_loss(y,p,labels=LABELS)),
            'brier_multiclass':br}

# Choose one shared C from mean 2023 Macro-F1 across the five logistic configurations.
def selected_common_C(s):
    pivot=s.groupby('C')['macro_f1'].mean().sort_index()
    return float(sorted(pivot.index,key=lambda c:(-pivot[c],c))[0])

# First fit uses 2022 only, with 2023 reserved for internal hyperparameter validation.
factory_dev=InputFactory(with_ticker=True).fit(train22)
x22=factory_dev.transform(train22)
x23=factory_dev.transform(val23)
tuning=[]
for c in CS:
    for model in MODEL_NAMES:
        clf=fit_model(x22[model],train22.target_label,c)
        yp,p=predict(clf,x23[model])
        tuning.append({'C':c,'model':model,**score(val23.target_label,yp,p)})
tuning=pd.DataFrame(tuning)
SELECTED_C=selected_common_C(tuning)
print('SHARED REGULARISATION SELECTED ONLY ON 2023:',SELECTED_C)
# Refit all transformations and five models on eligible 2022–2023 rows; evaluate on 2024.
factory=InputFactory(with_ticker=True).fit(dev)
xdev=factory.transform(dev)
xte=factory.transform(test)
models={}
predictions={}
metric_rows=[]
for name in MODEL_NAMES:
    clf=fit_model(xdev[name],dev.target_label,SELECTED_C)
    yp,p=predict(clf,xte[name]);models[name]=clf
    predictions[name]=pd.DataFrame({'date':test.date.to_numpy(),'ticker':test.ticker.to_numpy(),
        'actual':test.target_label.to_numpy(),'predicted':yp,
        'p_down':p[:,0],'p_flat':p[:,1],'p_up':p[:,2]})
    metric_rows.append({'model':name,'selected_C':SELECTED_C,**score(test.target_label,yp,p)})
# Nonlinear benchmark: Extra Trees uses the availability-control feature block, without text.
# Tune Extra Trees leaf size using 2023 validation only; retain the same 2024 test set.
tree_tuning=[]
for leaf in [3,8]:
    extra=ExtraTreesClassifier(n_estimators=80,max_depth=14,min_samples_leaf=leaf,max_features=.7,
                               random_state=SEED,n_jobs=2)
    extra.fit(x22['A_availability'],train22.target_label)
    yp,p=predict(extra,x23['A_availability'])
    tree_tuning.append({'C':np.nan,'model':'F_extra_trees','min_samples_leaf':leaf,
                        **score(val23.target_label,yp,p)})
tree_scores=pd.DataFrame(tree_tuning)
TREE_LEAF=int(tree_scores.sort_values(['macro_f1','min_samples_leaf'],ascending=[False,True]).iloc[0].min_samples_leaf)
tuning=pd.concat([tuning,tree_scores],ignore_index=True)
extra_final=ExtraTreesClassifier(n_estimators=80,max_depth=14,min_samples_leaf=TREE_LEAF,max_features=.7,
                                 random_state=SEED,n_jobs=2)
extra_final.fit(xdev['A_availability'],dev.target_label)
extra_yp,extra_p=predict(extra_final,xte['A_availability'])
models['F_extra_trees']=extra_final
predictions['F_extra_trees']=pd.DataFrame({'date':test.date.to_numpy(),'ticker':test.ticker.to_numpy(),
    'actual':test.target_label.to_numpy(),'predicted':extra_yp,
    'p_down':extra_p[:,0],'p_flat':extra_p[:,1],'p_up':extra_p[:,2]})
metrics=pd.DataFrame(metric_rows+[{'model':'F_extra_trees','selected_C':np.nan,
                                   'selected_min_samples_leaf':TREE_LEAF,**score(test.target_label,extra_yp,extra_p)}])
assert all(len(v)==3540 for v in predictions.values())
print(metrics[['model','macro_f1','log_loss','brier_multiclass']].to_string(index=False))

## 4. Produce the auditable sample, model, and subgroup tables

Tables **01–05** document source coverage, eligibility, target composition, threshold sensitivity, and file-integrity checks. Tables **06–07** document 2023-only hyperparameter comparison and selection. Table **08** compares all six model configurations on the same 3,540 eligible 2024 predictions. Tables **09–15** examine per-class errors, confusion counts, per-ticker variation, archived-text availability, current-session trading activity, the two halves of 2024, and seven-bin top-label calibration.

**How to interpret model rankings.** A higher Macro-F1 can accompany a worse log loss or Brier score. Treat the three model families separately where appropriate: the five logistic specifications support matched feature-ablation comparisons; Extra Trees checks sensitivity to estimator choice. Per-ticker, subgroup, and H1/H2 metrics are diagnostic descriptions, not new independent confirmation datasets.

**Calibration caution.** Top-label expected calibration error (ECE) is computed from seven equal-width confidence bins. ECE depends on the bin definition, sample size, and observations close to a bin boundary. The exported bin table includes denominators so that an apparent calibration difference can be audited rather than inferred from a single summary number. The committed tables exclude raw tweet text and observation-level prediction records.

In [ ]:
# Export deterministic, numbered analytical tables without source tweet text.
def write_table(number,slug,df):
    assert isinstance(df,pd.DataFrame) and len(df)>0,slug
    path=TAB/f'Table_{number:02d}_{slug}.csv'
    df.to_csv(path,index=False,float_format='%.10g')
    return df

# 01–04: annual and ticker coverage, eligible sample composition and target sensitivity.
annual=[]
for yr,g in panel.groupby('year'):
    elig=data[data.year.eq(yr)]
    annual.append({'year':yr,'market_sessions':len(g),'eligible':len(elig),
                   'excluded':len(g)-len(elig),'sessions_with_archived_text':int(g.has_archived_text.sum()),
                   'text_coverage_pct':100*g.has_archived_text.mean(),
                   'assigned_tweets':int(g.tweet_count.sum()),
                   'zero_volume_market_sessions':int((g.volume==0).sum())})
t01=write_table(1,'annual_sample_coverage',pd.DataFrame(annual))
ticker_annual=[]
for (tk,year),g in panel.groupby(['ticker','year']):
    ticker_annual.append({'ticker':tk,'year':year,'market_sessions':len(g),
         'eligible':int(g.eligible.sum()),'text_sessions':int(g.has_archived_text.sum()),
         'text_coverage_pct':100*g.has_archived_text.mean(),'assigned_tweets':int(g.tweet_count.sum()),
         'zero_volume_sessions':int((g.volume==0).sum())})
t02=write_table(2,'ticker_year_coverage',pd.DataFrame(ticker_annual))
t03=write_table(3,'target_class_by_year',data.groupby(['year','target_label'],observed=True).size().rename('n').reset_index().assign(
    proportion=lambda d:d.n/d.groupby('year').n.transform('sum')))
threshold=[]
for band in [EPS,.005,.01]:
    for cls in LABELS:
        y=np.where(dev.next_return < -band,'Down',np.where(dev.next_return >band,'Up','Flat'))
        threshold.append({'band_abs_return':band,'label':cls,'count':int(np.sum(y==cls)),
                           'share_pct':100*np.mean(y==cls),'basis':'development_2022_2023'})
t04=write_table(4,'threshold_sensitivity_development',pd.DataFrame(threshold))
# 05: audit directly observed source properties, joins, chronological split and file checksums.
checks=[
 ('Market rows',len(market),10830),('Tweet rows',len(tweets),7672),
 ('Assigned posts',len(assigned),7471),('Unassigned posts (quarantined)',len(unassigned),201),
 ('Posts assigned to 2022-2023',int((assigned.trading_session_date.dt.year<2024).sum()),2562),
 ('Posts assigned to 2024',int((assigned.trading_session_date.dt.year==2024).sum()),4909),
 ('Posts published 2023 / assigned 2024',len(cross_year),15),
 ('Ticker-date duplicates',int(market[['ticker','date']].duplicated().sum()),0),
 ('Eligible development rows',len(dev),6945),('Eligible 2024 rows',len(test),3540),
 ('2024 ticker count',test.ticker.nunique(),15),
 ('Development ends before test',int(dev.date.max()<test.date.min()),1),
 ('Market SHA256 matches verified input',int(sha256(MARKET_FILE)==EXPECTED_MARKET_SHA),1),
 ('Tweet SHA256 matches verified input',int(sha256(TWEET_FILE)==EXPECTED_TWEET_SHA),1),
 ('2024 classification output rows per model',min(len(v) for v in predictions.values()),3540),
 ('No label/future feature names',int(not any('next_' in x or 'target' in x for x in FEATURES)),1)]
qa=pd.DataFrame([{'check':k,'observed':o,'expected':e,'pass':o==e} for k,o,e in checks]);
qa=pd.concat([qa,pd.DataFrame([
    {'check':'SHA256 source market', 'observed':sha256(MARKET_FILE),'expected':EXPECTED_MARKET_SHA,'pass':True},
    {'check':'SHA256 source tweets','observed':sha256(TWEET_FILE),'expected':EXPECTED_TWEET_SHA,'pass':True},
    {'check':'Python version','observed':platform.python_version(),'expected':'environment-dependent','pass':True},
    {'check':'Scikit-learn version','observed':SKLEARN_VERSION,'expected':'document run version','pass':True}])],ignore_index=True)
assert qa['pass'].all()
t05=write_table(5,'source_integrity_and_provenance',qa)
# 06–08: export internal model selection and common-sample 2024 model performance.
t06=write_table(6,'development_2023_validation_C_search',tuning)
t07=write_table(7,'selected_development_model',pd.concat([
    tuning.loc[tuning.C.eq(SELECTED_C)].assign(selected_C=SELECTED_C,selection_rule='shared logistic C, 2023 validation'),
    tree_scores.loc[tree_scores.min_samples_leaf.eq(TREE_LEAF)].assign(selected_C=np.nan,selection_rule='best Extra Trees leaf size, 2023 validation')],ignore_index=True))
t08=write_table(8,'chronological_2024_model_comparison',metrics)

# 09–15: class-level errors, sample subgroups and top-label calibration bins.
cls_rows=[];cm_rows=[];ticker_rows=[];sub_text=[];sub_vol=[];sub_half=[];calrows=[]
# ECE bin edges cover [0,1], with empty bins retained for denominator transparency.
def ece_bins(frame,model,group='all',n_bins=7):
    probs=frame[['p_down','p_flat','p_up']].to_numpy()
    conf=probs.max(axis=1)
    correct=(frame.actual.to_numpy()==frame.predicted.to_numpy()).astype(float)
    bins=np.linspace(0,1,n_bins+1);idx=np.minimum(np.searchsorted(bins,conf,side='right')-1,n_bins-1)
    out=[]
    for b in range(n_bins):
        mask=idx==b
        out.append({'model':model,'group':group,'bin':b+1,'bin_lower':bins[b],
                    'bin_upper':bins[b+1],'n':int(mask.sum()),
                    'confidence':float(conf[mask].mean()) if mask.sum() else np.nan,
                    'observed_accuracy':float(correct[mask].mean()) if mask.sum() else np.nan,
                    'abs_gap':float(abs(correct[mask].mean()-conf[mask].mean())) if mask.sum() else np.nan})
    return out
for name,d in predictions.items():
    y=d.actual.to_numpy();yp=d.predicted.to_numpy()
    precision,recall,f1,support=precision_recall_fscore_support(y,yp,labels=LABELS,zero_division=0)
    for j,l in enumerate(LABELS):cls_rows.append({'model':name,'class':l,'support':int(support[j]),
                        'precision':precision[j],'recall':recall[j],'f1':f1[j]})
    c=confusion_matrix(y,yp,labels=LABELS)
    for a,l1 in enumerate(LABELS):
        for b,l2 in enumerate(LABELS):cm_rows.append({'model':name,'actual':l1,'predicted':l2,'n':int(c[a,b])})
    for tk,g in d.groupby('ticker'):
        pp=g[['p_down','p_flat','p_up']].to_numpy()
        ticker_rows.append({'model':name,'ticker':tk,**score(g.actual,g.predicted,pp)})
    dd=d.merge(test[['ticker','date','has_archived_text','zero_volume']],on=['ticker','date'],validate='one_to_one')
    for col,bag in [('has_archived_text',sub_text),('zero_volume',sub_vol)]:
        for grp,group in dd.groupby(col):
            pp=group[['p_down','p_flat','p_up']].to_numpy()
            bag.append({'model':name,'subgroup':col,'value':int(grp),**score(group.actual,group.predicted,pp)})
    for half,group in d.groupby(d.date.dt.month.gt(6).map({False:'H1',True:'H2'})):
        pp=group[['p_down','p_flat','p_up']].to_numpy()
        sub_half.append({'model':name,'halfyear':half,**score(group.actual,group.predicted,pp)})
    calrows.extend(ece_bins(d,name))
t09=write_table(9,'classwise_2024',pd.DataFrame(cls_rows))
t10=write_table(10,'confusion_matrices_2024_long',pd.DataFrame(cm_rows))
t11=write_table(11,'ticker_2024_performance',pd.DataFrame(ticker_rows))
t12=write_table(12,'text_availability_subgroups_2024',pd.DataFrame(sub_text))
t13=write_table(13,'volume_subgroups_2024',pd.DataFrame(sub_vol))
t14=write_table(14,'2024_first_second_half_performance',pd.DataFrame(sub_half))
calibration=pd.DataFrame(calrows)
t15=write_table(15,'calibration_bins_2024',calibration)

## 5. Quantify uncertainty and test lexical masking sensitivity

**Paired date-block bootstrap.** Resample consecutive sequences of trading dates, retaining the full cross-section of 15 tickers for every sampled date. This preserves within-date dependence between ticker outcomes and creates paired contrasts because every model uses the **same resampled date indices**. Run 2,000 circular block-bootstrap replicates for block lengths of **5, 10, and 20 trading sessions**. Report the observed difference, percentile 95% interval, and the recorded centered-bootstrap two-sided p-value.

**Multiplicity definition.** Holm adjustment is applied to the **three lexical-versus-availability Macro-F1 contrasts** (word, character, and hybrid) *separately for each block length*. It does not retrospectively correct every model, subgroup, threshold choice, or research decision. Intervals are conditional on the fitted predictions; they do **not** account for model-refitting or hyperparameter-selection uncertainty. A CI containing zero does not substantiate consistent lexical superiority.

**Masking diagnostic.** Transform 2024 inputs again after replacing their lexical documents with empty strings, while leaving observed market predictors and availability indicators unchanged. Apply the **already fitted** character model without refitting it. Differences indicate a model-level sensitivity to the supplied lexical block under this perturbation. This is not a causal estimate of social-media text effects or a valid simulation of a truly missing tweet archive.

In [ ]:
# Preserve daily cross-sectional dependence when constructing paired resampling blocks.
dates=np.sort(test.date.unique());nd=len(dates)
assert nd==236
byday={}
for model,frame in predictions.items():
    confusion=np.zeros((nd,3,3),dtype=np.int64)
    ll=np.zeros(nd,dtype=float);brier=np.zeros(nd,dtype=float)
    for j,day in enumerate(dates):
        g=frame[frame.date.eq(day)];p=g[['p_down','p_flat','p_up']].to_numpy()
        code=pd.Categorical(g.actual,categories=LABELS).codes
        confusion[j]=confusion_matrix(g.actual,g.predicted,labels=LABELS)
        ll[j]=-np.log(np.maximum(p[np.arange(len(g)),code],1e-15)).sum()
        brier[j]=np.sum((p-np.eye(3)[code])**2)
    byday[model]={'cm':confusion,'loss':ll,'brier':brier}

# Convert the selected confusion matrices to vectorised three-class Macro-F1 estimates.
def macro_from_cm(mats):
    mat=np.asarray(mats)
    tp=np.diagonal(mat,axis1=-2,axis2=-1)
    denom=mat.sum(axis=-1)+mat.sum(axis=-2)
    f=np.divide(2*tp,denom,out=np.zeros_like(tp,dtype=float),where=denom!=0)
    return f.mean(axis=-1)

# Holm step-down procedure for a prespecified family of p-values.
def holm(ps):
    order=np.argsort(ps);out=np.zeros(len(ps));prior=0
    for j,ix in enumerate(order):
        prior=max(prior,min(1.,(len(ps)-j)*float(ps[ix])))
        out[ix]=prior
    return out

# Compare lexical variants against the same availability control, not market-only.
contrasts=['W_word','C_character','H_hybrid']
bootrows=[]
# Circular moving blocks resample trading dates; identical indices are used for every model.
for L in [5,10,20]:
    rng=np.random.default_rng(SEED+L)
    nstarts=int(np.ceil(nd/L))
    starts=rng.integers(0,nd,size=(BS_REPS,nstarts))
    ids=((starts[:,:,None]+np.arange(L)[None,None,:])%nd).reshape(BS_REPS,-1)[:,:nd]
    bmetrics={}
    for m in ALL_MODELS:
        q=byday[m]
        cm=q['cm'][ids].sum(axis=1)
        bmetrics[m]={'macro_f1':macro_from_cm(cm),
              'log_loss':q['loss'][ids].sum(axis=1)/(nd*15),
              'brier_multiclass':q['brier'][ids].sum(axis=1)/(nd*15)}
    for m in ['M_market','F_extra_trees']+contrasts:
        for metric in ['macro_f1','log_loss','brier_multiclass']:
            diff=bmetrics[m][metric]-bmetrics['A_availability'][metric]
            observed=(metrics.set_index('model').loc[m,metric]-
                      metrics.set_index('model').loc['A_availability',metric])
# Centered bootstrap p-values and percentile intervals condition on fixed fitted predictions.
            p_centered=min(1.,2*min(np.mean(diff-observed<=-abs(observed)),
                                      np.mean(diff-observed>=abs(observed))))
            bootrows.append({'block_length':L,'contrast':m+' minus A_availability','metric':metric,
                    'observed_difference':observed,'ci_low':np.quantile(diff,.025),
                    'ci_high':np.quantile(diff,.975),'p_centered_two_sided':p_centered,
                    'bootstrap_reps':BS_REPS})
bootstrap=pd.DataFrame(bootrows)
t16=write_table(16,'date_block_bootstrap_2024',bootstrap)
# Adjust only three lexical Macro-F1 tests, independently within each block length.
multiple=[]
for L in [5,10,20]:
    sub=bootstrap[(bootstrap.block_length.eq(L))&(bootstrap.metric.eq('macro_f1'))&
                  (bootstrap.contrast.isin([x+' minus A_availability' for x in contrasts]))].copy()
    adj=holm(sub.p_centered_two_sided.to_numpy())
    for j,(_,r) in enumerate(sub.iterrows()):
        multiple.append({'block_length':L,'contrast':r.contrast,'effect_macro_f1':r.observed_difference,
           'CI95_low':r.ci_low,'CI95_high':r.ci_high,'p_uncorrected':r.p_centered_two_sided,
           'p_holm_three_contrasts':adj[j],
           'reject_at_005':bool(adj[j]<.05 and r.ci_low>0)})
t17=write_table(17,'holm_adjusted_lexical_tests',pd.DataFrame(multiple))
# Mask evaluation text without changing availability or fitting any new model parameters.
xte_mask=factory.transform(test,mask_text=True)
masked_predictions=predictions['C_character']
masked_yp,masked_p=predict(models['C_character'],xte_mask['C_character'])
maskrow=[{'test_condition':'Observed archived text',**score(test.target_label,
                       masked_predictions.predicted,masked_predictions[['p_down','p_flat','p_up']].to_numpy())},
         {'test_condition':'Mask lexical text, preserve availability',**score(test.target_label,masked_yp,masked_p)}]
t18=write_table(18,'character_text_masking_sensitivity',pd.DataFrame(maskrow))
print('BOOTSTRAP AND TEXT MASKING COMPLETE')

## 6. Test generalisation across excluded stocks

**Purpose.** The chronological 2024 test contains stocks represented during development. The additional stock-exclusion experiment asks whether associations transfer to stocks whose **supervised labels** were excluded from both parameter selection and final training.

**Procedure.** Sort the 15 tickers and divide them into three predetermined groups of five. For each fold, fit on the other 10 stocks, validate hyperparameters using their 2023 records, then refit on their eligible 2022–2023 observations. Evaluate the five excluded stocks in 2024. Remove ticker-identity one-hot indicators in this experiment because unseen ticker categories cannot contribute learned stock-specific coefficients. Compare the market-only, availability-control, and character-text specifications.

**Interpretation.** Pool non-overlapping held-out ticker predictions only after confirming 3,540 distinct 2024 ticker–date keys. These are **within-archive stock-transfer** results, not validation on a new market, a new data provider, or an independent population. Historical prices and tweets for held-out tickers are allowed as time-available predictors; their labels do not affect supervised fitting or selection.

In [ ]:
# Assign deterministic, mutually exclusive ticker groups before scoring any 2024 targets.
all_tickers=sorted(dev.ticker.unique().tolist())
folds=[all_tickers[0:5],all_tickers[5:10],all_tickers[10:15]]
foldrows=[]
pooled_predictions={k:[] for k in ['M_market','A_availability','C_character']}
# Remove every held-out ticker from supervised training and 2023 model selection.
for k,held in enumerate(folds,1):
    retained=[z for z in all_tickers if z not in held]
    tr=train22[train22.ticker.isin(retained)]
    va=val23[val23.ticker.isin(retained)]
    dov=dev[dev.ticker.isin(retained)]
    ev=test[test.ticker.isin(held)]
# Fit fresh preprocessing objects using only the ten retained training stocks.
    fac1=InputFactory(with_ticker=False).fit(tr)
    xtr=fac1.transform(tr); xval=fac1.transform(va)
    cross_val=[]
    active=['M_market','A_availability','C_character']
    for c in CS:
        for model in active:
            yp,p=predict(fit_model(xtr[model],tr.target_label,c),xval[model])
            cross_val.append({'C':c,'model':model,**score(va.target_label,yp,p)})
    chosen=selected_common_C(pd.DataFrame(cross_val))
# Refit after 2023 selection on retained 2022–2023 data and score held-out 2024 stocks.
    fac2=InputFactory(with_ticker=False).fit(dov)
    ftr=fac2.transform(dov);fev=fac2.transform(ev)
    for model in active:
        pmodel=fit_model(ftr[model],dov.target_label,chosen)
        yp,prob=predict(pmodel,fev[model])
        foldrows.append({'fold':k,'heldout_tickers':','.join(held),'model':model,
                   'n_train':len(dov),'n_2024_test':len(ev),'selected_C':chosen,
                   **score(ev.target_label,yp,prob)})
        pooled_predictions[model].append(pd.DataFrame({'date':ev.date.to_numpy(),
                  'ticker':ev.ticker.to_numpy(),'actual':ev.target_label.to_numpy(),
                  'predicted':yp,'p_down':prob[:,0],'p_flat':prob[:,1],'p_up':prob[:,2]}))
    print('Ticker-disjoint fold',k,'held out',','.join(held),'C',chosen)
t19=write_table(19,'ticker_disjoint_fold_results',pd.DataFrame(foldrows))
# Pool disjoint held-out predictions only after checking coverage and key uniqueness.
pooled=[]
for model,parts in pooled_predictions.items():
    whole=pd.concat(parts,ignore_index=True).sort_values(['date','ticker']).reset_index(drop=True)
    assert len(whole)==3540 and whole[['ticker','date']].duplicated().sum()==0
    pooled.append({'model':model,**score(whole.actual,whole.predicted,
                                whole[['p_down','p_flat','p_up']].to_numpy())})
t20=write_table(20,'ticker_disjoint_pooled_results',pd.DataFrame(pooled))
print('HOLDOUT POOLED',t20[['model','macro_f1','log_loss']].to_string(index=False))

## 7. Render publication-ready figures from the computed tables

**Visual evidence contract.** Create exactly 20 sequentially numbered 300-dpi PNGs using the 20 CSV outputs generated by this run. Use stable class labels, legible captions, numerical axes, and a colour-blind-conscious palette. Do not substitute illustrative data or modify model scores to improve presentation.

**Figure families.** Figures 01–05 describe data coverage, eligibility, class composition, and target sensitivity. Figures 06–11 show development selection and test-set model rankings across classification and probability metrics. Figures 12–17 show class recall, confusion counts, calibration and per-ticker/subgroup heterogeneity. Figures 18–19 show uncertainty intervals and sensitivity to bootstrap block length. Figure 20 summarises stock-exclusion pooled Macro-F1.

**Evidence interpretation.** Figure 07 compares descriptive 2024 point estimates; Figure 18 supplies uncertainty for the paired lexical contrasts. Figure 14 is based on seven top-label calibration bins and can be unstable when bins contain few observations. Figure 20 represents a different training design from Figure 07 and must not be read as a direct single-model ablation. Figure headings are designed for inspection; journal artwork may later move these headings into external captions without altering the underlying data.

In [ ]:
# All visualisations are generated from the corresponding analytical tables.
def save_figure(num,name,fig):
    fig.savefig(FIG/f'Figure_{num:02d}_{name}.png',dpi=300,bbox_inches='tight',facecolor='white')
    plt.close(fig)

def newfig(w=8,h=4):
    return plt.subplots(figsize=(w,h),dpi=110)

def label_values(ax,fmt='{:.3f}',horizontal=False):
    for bar in ax.patches:
        if horizontal:
            ax.text(bar.get_width()+.003,bar.get_y()+bar.get_height()/2,
                    fmt.format(bar.get_width()),va='center',fontsize=8)
        else:
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.005,
                    fmt.format(bar.get_height()),ha='center',fontsize=8)

def tidy(ax,title,xlabel='',ylabel=''):
    ax.set_title(title,pad=10);ax.set_xlabel(xlabel);ax.set_ylabel(ylabel)
    ax.grid(axis='y',alpha=.13)
    ax.set_axisbelow(True)

# Figure 01. Annual archive-presence rate, using observed ticker-session denominators.
fig,ax=newfig(7,3.7)
ax.bar(t01.year.astype(str),t01.text_coverage_pct,color=COLORS[:3]);
for i,v in enumerate(t01.text_coverage_pct):ax.text(i,v+.8,f'{v:.1f}%',ha='center')
ax.set_ylim(0,60);tidy(ax,'Observed archive text coverage','Trading year','Ticker-sessions with ≥1 archived tweet (%)')
fig.tight_layout();save_figure(1,'annual_text_coverage',fig)
# Figure 02. Heterogeneity in the archive-presence rate across stock and year.
hm=t02.pivot(index='ticker',columns='year',values='text_coverage_pct').sort_index()
fig,ax=newfig(7,6)
im=ax.imshow(hm.to_numpy(),aspect='auto',vmin=0,vmax=100,cmap='viridis')
ax.set_xticks(range(len(hm.columns)),hm.columns);ax.set_yticks(range(len(hm.index)),hm.index)
ax.set_title('Archive coverage by stock and trading year',pad=12)
for (i,j),v in np.ndenumerate(hm.to_numpy()):ax.text(j,i,f'{v:.0f}',ha='center',va='center',fontsize=7,
                                         color='white' if v<40 else 'black')
fig.colorbar(im,ax=ax,label='Coverage (%)');fig.tight_layout();save_figure(2,'ticker_year_coverage_heatmap',fig)
# Figure 03. Eligibility losses from warm-up and end-of-year target filtering.
fig,ax=newfig(7.5,3.8);xx=np.arange(len(t01));ax.bar(xx,t01.eligible,color=COLORS[0],label='Eligible')
ax.bar(xx,t01.excluded,bottom=t01.eligible,color=COLORS[1],label='Excluded')
ax.set_xticks(xx,t01.year.astype(str));ax.legend();tidy(ax,'Sample eligibility by year','Trading year','Market observations')
fig.tight_layout();save_figure(3,'eligible_excluded_observations',fig)
# Figure 04. Within-year Down, Flat and Up class shares for eligible observations.
classwide=t03.pivot(index='year',columns='target_label',values='proportion').reindex(columns=LABELS)
fig,ax=newfig(7.2,3.8)
bot=np.zeros(len(classwide))
for j,label in enumerate(LABELS):
    vv=classwide[label].to_numpy()*100
    ax.bar(classwide.index.astype(str),vv,bottom=bot,label=label,color=[COLORS[0],COLORS[1],COLORS[2]][j])
    for i,v in enumerate(vv):ax.text(i,bot[i]+v/2,f'{v:.1f}%',ha='center',va='center',fontsize=8)
    bot+=vv
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=3)
tidy(ax,'Next-session class composition','Trading year','Eligible sample share (%)');fig.tight_layout();save_figure(4,'class_distribution_by_year',fig)
# Figure 05. Target composition under numerical zero, ±0.5% and ±1.0% bands.
classsens=t04.pivot(index='band_abs_return',columns='label',values='share_pct').reindex(columns=LABELS)
fig,ax=newfig(7.4,3.8)
xx=np.arange(len(classsens));w=.25
for j,name in enumerate(LABELS):ax.bar(xx+(j-1)*w,classsens[name],w,label=name,color=COLORS[j])
ax.set_xticks(xx,['Numerical zero','±0.5%','±1.0%']);ax.legend(ncol=3)
tidy(ax,'Development target sensitivity (same observations)','Return interval for Flat','Class share (%)')
fig.tight_layout();save_figure(5,'development_threshold_sensitivity',fig)
# Figure 06. Candidate C values evaluated on 2023, before any 2024 scoring.
fig,ax=newfig(8,4.4)
for j,model in enumerate(MODEL_NAMES):
    g=t06[t06.model.eq(model)].sort_values('C')
    ax.plot(np.arange(len(g)),g.macro_f1,marker='o',label=MODEL_LABELS[model],color=COLORS[j])
ax.axvline(CS.index(SELECTED_C),ls='--',c='gray',lw=1,label='Selected C')
ax.set_xticks(np.arange(len(CS)),[str(x) for x in CS]);ax.legend(fontsize=8,ncol=2)
tidy(ax,'Development selection on 2023 only','Regularisation parameter C','Validation Macro-F1')
fig.tight_layout();save_figure(6,'internal_2023_validation',fig)
# Figure 07. Descriptive 2024 Macro-F1 comparison, not a significance test.
mm=t08.set_index('model').loc[ALL_MODELS]
fig,ax=newfig(8,4);ax.barh(range(len(ALL_MODELS)),mm.macro_f1,color=COLORS)
ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS]);ax.invert_yaxis();ax.set_xlim(0,.6)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Chronological evaluation: 2024','Macro-F1 (higher is better)','')
fig.tight_layout();save_figure(7,'2024_macro_f1',fig)
# Figure 08. Balanced accuracy averages recall across Down, Flat and Up.
fig,ax=newfig(8,4);ax.barh(range(len(ALL_MODELS)),mm.balanced_accuracy,color=COLORS)
ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS]);ax.invert_yaxis();ax.set_xlim(0,.6)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Chronological evaluation: class-balanced accuracy','Balanced accuracy (higher is better)','')
fig.tight_layout();save_figure(8,'2024_balanced_accuracy',fig)
# Figure 09. Multiclass log loss; show the full baseline to avoid magnifying small changes.
fig,ax=newfig(8,4);ax.barh(range(len(ALL_MODELS)),mm.log_loss,color=COLORS)
ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS]);ax.invert_yaxis();ax.set_xlim(0,1.10)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Multiclass log loss: 2024','Log loss (lower is better)','')
fig.tight_layout();save_figure(9,'2024_multiclass_log_loss',fig)
# Figure 10. Three-class Brier score using the documented summed convention.
fig,ax=newfig(8,4);ax.barh(range(len(ALL_MODELS)),mm.brier_multiclass,color=COLORS)
ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS]);ax.invert_yaxis();ax.set_xlim(0,.7)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Multiclass Brier score: 2024','Brier score (lower is better)','')
fig.tight_layout();save_figure(10,'2024_multiclass_brier',fig)
# Figure 11. Seven-bin top-label ECE; lower values alone do not prove calibration.
x=t15.dropna(subset=['abs_gap']).assign(weight=lambda z:z.n/3540)
ece=x.assign(contrib=lambda z:z.abs_gap*z.weight).groupby('model').contrib.sum().reindex(ALL_MODELS)
fig,ax=newfig(8,4);ax.barh(range(len(ALL_MODELS)),ece,color=COLORS)
ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS]);ax.invert_yaxis();ax.set_xlim(0,max(ece)*1.35)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Top-label expected calibration error: 7 bins','ECE (lower is better)','')
fig.tight_layout();save_figure(11,'2024_calibration_error',fig)
# Figure 12. Recall by class and model; inspect asymmetry in the Up class.
recm=t09.pivot(index='model',columns='class',values='recall').reindex(index=ALL_MODELS,columns=LABELS)
fig,ax=newfig(7.4,4)
im=ax.imshow(recm.to_numpy(),cmap='viridis',vmin=0,vmax=1,aspect='auto')
ax.set_xticks(range(3),LABELS);ax.set_yticks(range(len(ALL_MODELS)),[MODEL_LABELS[m] for m in ALL_MODELS])
for (i,j),v in np.ndenumerate(recm.to_numpy()):ax.text(j,i,f'{v:.2f}',ha='center',va='center',color='white' if v<.5 else 'black')
fig.colorbar(im,ax=ax,label='Recall');ax.set_title('Class recall, same 2024 sample')
fig.tight_layout();save_figure(12,'class_recall_heatmap',fig)
# Figure 13. Unnormalised confusion matrix for the character-text model.
cm=t10[t10.model.eq('C_character')].pivot(index='actual',columns='predicted',values='n').reindex(index=LABELS,columns=LABELS)
fig,ax=newfig(5.6,4.8);im=ax.imshow(cm.to_numpy(),cmap='Blues')
ax.set_xticks(range(3),LABELS);ax.set_yticks(range(3),LABELS)
for (i,j),v in np.ndenumerate(cm.to_numpy()):ax.text(j,i,str(int(v)),va='center',ha='center',color='white' if v>cm.to_numpy().max()*.55 else 'black')
ax.set_xlabel('Predicted');ax.set_ylabel('Actual');ax.set_title('Character TF-IDF: 2024 confusion counts')
fig.colorbar(im,ax=ax,label='Observations');fig.tight_layout();save_figure(13,'character_confusion_matrix',fig)
# Figure 14. Confidence versus empirical accuracy, with possible sparse high bins.
fig,ax=newfig(6.4,5.4);ax.plot([0,1],[0,1],ls='--',lw=1,c='gray',label='Perfect')
for j,model in enumerate(ALL_MODELS):
    g=t15[(t15.model.eq(model))&(t15.n.gt(0))]
    ax.plot(g.confidence,g.observed_accuracy,marker='o',label=MODEL_LABELS[model],color=COLORS[j])
ax.legend(fontsize=7,loc='upper left');ax.set_xlim(0,1);ax.set_ylim(0,1)
tidy(ax,'Top-label reliability (7 equal-width bins)','Predicted top-label confidence','Observed accuracy')
fig.tight_layout();save_figure(14,'2024_top_label_reliability',fig)
# Figure 15. Per-ticker character-minus-availability Macro-F1 differences.
bytk=t11.pivot(index='ticker',columns='model',values='macro_f1')
bytk['difference']=bytk.C_character-bytk.A_availability
fig,ax=newfig(9,4.8);v=bytk.difference.sort_values()
ax.bar(v.index,v.values,color=[COLORS[2] if z>=0 else COLORS[3] for z in v]);ax.axhline(0,c='gray',lw=1)
ax.tick_params(axis='x',labelrotation=60);tidy(ax,'Character minus availability: per-stock Macro-F1','Ticker','Paired descriptive difference')
fig.tight_layout();save_figure(15,'ticker_level_macro_f1_delta',fig)
# Figure 16. Descriptive performance conditional on whether the archive contains text.
fig,ax=newfig(9,4.1);pivot=t12.pivot(index='model',columns='value',values='macro_f1').reindex(ALL_MODELS)
xx=np.arange(len(ALL_MODELS));w=.36
ax.bar(xx-w/2,pivot[0].to_numpy(),w,label='No archived tweet',color=COLORS[0]);ax.bar(xx+w/2,pivot[1].to_numpy(),w,label='At least one tweet',color=COLORS[1])
ax.set_xticks(xx,['Market','Availability','Word','Character','Hybrid','Extra Trees'],rotation=18);ax.set_ylim(0,.65);ax.legend()
tidy(ax,'Model performance conditional on archived text','Model','Macro-F1 within subgroup')
fig.tight_layout();save_figure(16,'availability_subgroup_macro_f1',fig)
# Figure 17. Diagnostic split by current-session trading volume.
fig,ax=newfig(9,4.1);pivot=t13.pivot(index='model',columns='value',values='macro_f1').reindex(ALL_MODELS)
ax.bar(xx-w/2,pivot[0].to_numpy(),w,label='Positive volume',color=COLORS[0])
ax.bar(xx+w/2,pivot[1].to_numpy(),w,label='Zero volume',color=COLORS[3])
ax.set_xticks(xx,['Market','Availability','Word','Character','Hybrid','Extra Trees'],rotation=18);ax.set_ylim(0,.7);ax.legend()
tidy(ax,'Sensitivity to trading inactivity','Model','Macro-F1 within subgroup')
fig.tight_layout();save_figure(17,'volume_subgroup_macro_f1',fig)
# Figure 18. Paired 95% bootstrap intervals using 10-session date blocks.
fig,ax=newfig(8,3.5)
g=t16[(t16.block_length.eq(10))&(t16.metric.eq('macro_f1'))&
          t16.contrast.isin([x+' minus A_availability' for x in contrasts])].set_index('contrast')
for i,m in enumerate(contrasts):
    r=g.loc[m+' minus A_availability'];x0=r.observed_difference
    ax.errorbar(x0,i,xerr=[[x0-r.ci_low],[r.ci_high-x0]],fmt='o',capsize=5,color=COLORS[i+1],ecolor=COLORS[i+1])
ax.axvline(0,ls='--',c='gray');ax.set_yticks(range(3),['Word','Character','Hybrid']);ax.invert_yaxis()
tidy(ax,'Paired block bootstrap, 10 sessions (2,000 repeats)','Δ Macro-F1 versus availability control','')
fig.tight_layout();save_figure(18,'paired_bootstrap_95ci',fig)
# Figure 19. Interval sensitivity to 5-, 10-, and 20-session blocks.
fig,ax=newfig(8,4.3)
for i,m in enumerate(contrasts):
    s=t16[(t16.metric.eq('macro_f1'))&(t16.contrast.eq(m+' minus A_availability'))].sort_values('block_length')
    ax.errorbar(s.block_length+(i-1)*.6,s.observed_difference,
        yerr=[s.observed_difference-s.ci_low,s.ci_high-s.observed_difference],fmt='o-',
        capsize=3,color=COLORS[i+1],label=MODEL_LABELS[m])
ax.axhline(0,ls='--',c='gray');ax.set_xticks([5,10,20]);ax.legend(fontsize=8)
tidy(ax,'Sensitivity of estimated lexical effects to block length','Date-block length (sessions)','Δ Macro-F1, 95% bootstrap CI')
fig.tight_layout();save_figure(19,'bootstrap_block_length_robustness',fig)
# Figure 20. Pool predictions from all mutually exclusive five-stock holdout folds.
fig,ax=newfig(8,3.6)
order=['M_market','A_availability','C_character'];po=t20.set_index('model').loc[order]
ax.barh(range(3),po.macro_f1,color=[COLORS[0],COLORS[1],COLORS[3]])
ax.set_yticks(range(3),[MODEL_LABELS[m] for m in order]);ax.invert_yaxis();ax.set_xlim(0,.6)
label_values(ax,'{:.4f}',horizontal=True)
tidy(ax,'Five-stock group exclusions: pooled 2024 evaluation','Macro-F1 (held-out stock labels excluded)','')
fig.tight_layout();save_figure(20,'ticker_disjoint_pooled_macro_f1',fig)

## 8. Verify all outputs and create the final ZIP

**Final reproducibility gate.** Require exactly 20 numbered, non-empty PNG files and 20 numbered, non-empty CSV files. Confirm the ZIP archive contains **only these 40 files**, organised as `figures/` and `tables/`; no original tweets, source market records, manuscript, or bibliographic exports enter the download package.

**Environment-specific delivery.** In Google Colab, import `files` from `google.colab` and call `files.download(path)` from Python. In Kaggle, display a notebook download link to the archive in `/kaggle/working/`. In local Jupyter, print the saved path. This avoids the undefined browser JavaScript function `download()`.

**Review before citation.** Inspect Table 05 for source identity, Table 07 for 2023 selection, Table 08 for 2024 scores, Tables 16–17 for uncertainty and multiplicity, and Tables 19–20 for stock transfer. Preserve the original model-selection rule, seeds, package versions, and archive hashes when publishing a new release. If results differ across environments, document the discrepancy rather than silently overwriting the reference artifacts.

In [ ]:
# Fail explicitly if any numbered analytical output is missing or empty.
pngs=sorted(FIG.glob('*.png'))
csvs=sorted(TAB.glob('*.csv'))
assert len(pngs)==20, f'Expected exactly 20 PNG; got {len(pngs)}'
assert len(csvs)==20, f'Expected exactly 20 CSV; got {len(csvs)}'
assert [int(x.name.split('_')[1]) for x in pngs]==list(range(1,21))
assert [int(x.name.split('_')[1]) for x in csvs]==list(range(1,21))
for p in pngs:
    assert p.stat().st_size>5000, f'PNG seems empty: {p.name}'
for p in csvs:
    assert p.stat().st_size>50 and len(pd.read_csv(p))>0, f'CSV seems empty: {p.name}'
# ZIP includes only the 20 figures and 20 CSV tables, not raw sources or manuscripts.
with zipfile.ZipFile(ZIP_FILE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for p in pngs+csvs:
        zf.write(p,arcname=f'{p.parent.name}/{p.name}')
with zipfile.ZipFile(ZIP_FILE) as zf:
    assert zf.testzip() is None
    items=zf.namelist()
    assert len(items)==40 and sum(p.endswith('.png') for p in items)==20 and sum(p.endswith('.csv') for p in items)==20
# Report the final archive path and test contrasts before platform-specific delivery.
print(f'VALIDATED: {len(pngs)} PNG at 300 dpi + {len(csvs)} CSV, 40 files in one ZIP')
print('RESULT:',ZIP_FILE)
print('DATA INTEGRITY:', 'verified SHA-256, 10,830 market rows, 7,672 tweets, 6,945 development, 3,540 evaluated')
print('STATS:',t17[t17.block_length.eq(10)][['contrast','effect_macro_f1','CI95_low','CI95_high','p_holm_three_contrasts']].to_string(index=False))
# Colab uses its Python download API; Kaggle uses a file link, not browser JavaScript.
if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_FILE))
elif IS_KAGGLE:
    from IPython.display import FileLink
    display(FileLink(str(ZIP_FILE),result_html_prefix='Download Kaggle ZIP: '))
else:
    print('LOCAL RUN: ZIP created. In Colab the final cell triggers download automatically.')